# Case study: the rare neutral class

*Session 9, block 3 practice. Author: course team, licence CC-BY-4.0.*

Only 7.5 % of the training reviews are neutral, but the neutral class counts one third in macro-F1. We compare five strategies inside imbalanced-learn pipelines, so that resampling happens only on the training folds:

no correction · random undersampling · random oversampling · SMOTE · class weights

The features are the leakage-free features of blocks 1 and 2. Validation is time-based (`TimeSeriesSplit` on the date-sorted sample). Theory: [03-class-imbalance.md](../theory/03-class-imbalance.md).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
DATA = ROOT / "case-study" / "data"
# product history from the FULL training table, in time order
train = pd.read_parquet(DATA / "train.parquet", columns=["review_id", "parent_asin", "user_id", "rating", "date"])
train = train.sort_values(["date", "review_id"]).reset_index(drop=True)
g = train.groupby("parent_asin")["rating"]
train["prod_prev_n"] = g.cumcount()
train["prod_prev_mean"] = ((g.cumsum() - train["rating"]) / train["prod_prev_n"]).where(train["prod_prev_n"] > 0)
train["user_prev_n"] = train.groupby("user_id").cumcount()
history = train[["review_id", "prod_prev_n", "prod_prev_mean", "user_prev_n"]]

reviews = pd.read_parquet(DATA / "train_sample.parquet")
df = reviews.merge(history, on="review_id", how="left").sort_values("date").reset_index(drop=True)
df["label"].value_counts(normalize=True).round(3)

In [ ]:
NEGATIONS = r"\b(?:not|no|never|don't|doesn't|didn't|isn't|wasn't|won't|can't)\b"


def text_stats(frame):
    """Seven numeric statistics of review text and title (Session 9, page 2)."""
    text, title = frame["text"].fillna(""), frame["title"].fillna("")
    n_chars = text.str.len()
    return pd.DataFrame({
        "log_chars": np.log1p(n_chars),
        "n_words": text.str.split().str.len(),
        "n_exclaim": text.str.count("!"),
        "n_question": text.str.count(r"\?"),
        "upper_share": text.str.count(r"[A-Z]") / n_chars.clip(lower=1),
        "n_negations": text.str.lower().str.count(NEGATIONS),
        "title_words": title.str.split().str.len(),
    }, index=frame.index)

X = pd.concat([
    text_stats(df),
    df[["verified_purchase", "helpful_vote", "n_images", "prod_prev_n", "prod_prev_mean", "user_prev_n"]].astype(float),
    pd.DataFrame({"year": df["date"].dt.year}),
], axis=1)
y = df["label"]
X.isna().mean().loc[lambda s: s > 0]   # prod_prev_mean is missing for first reviews

## 1. Five strategies with a logistic regression

SMOTE and logistic regression cannot handle missing values, so the pipeline imputes them (with an indicator column) and scales the features before resampling.

In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import make_pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

cv = TimeSeriesSplit(n_splits=5)
LABELS = ["neg", "neu", "pos"]


def prep():
    return [SimpleImputer(strategy="median", add_indicator=True), StandardScaler()]


def evaluate(models):
    """Time-based CV; per-class scores pooled over the five validation folds."""
    rows = []
    for name, model in models.items():
        y_true, y_pred = [], []
        for tr, va in cv.split(X):
            model.fit(X.iloc[tr], y.iloc[tr])                 # samplers act on the training fold only
            y_true.append(y.iloc[va])
            y_pred.append(model.predict(X.iloc[va]))
        y_true, y_pred = pd.concat(y_true), np.concatenate(y_pred)
        p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
        rows.append({"method": name, "accuracy": accuracy_score(y_true, y_pred), "macro_f1": f.mean(),
                     "neu_precision": p[1], "neu_recall": r[1], "neu_f1": f[1]})
    return pd.DataFrame(rows).set_index("method").round(3)


logreg = lambda cw=None: LogisticRegression(max_iter=2000, class_weight=cw)
linear = {
    "no correction": make_pipeline(*prep(), logreg()),
    "undersampling": make_pipeline(*prep(), RandomUnderSampler(random_state=0), logreg()),
    "oversampling": make_pipeline(*prep(), RandomOverSampler(random_state=0), logreg()),
    "SMOTE": make_pipeline(*prep(), SMOTE(random_state=0), logreg()),
    "class weights": make_pipeline(*prep(), logreg("balanced")),
}
res_linear = evaluate(linear)
res_linear

## 2. The same with gradient boosting

`HistGradientBoostingClassifier` handles missing values itself and supports `class_weight`. SMOTE still needs imputed values.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = lambda cw=None: HistGradientBoostingClassifier(class_weight=cw, random_state=0)
boosting = {
    "no correction": make_pipeline(hgb()),
    "undersampling": make_pipeline(RandomUnderSampler(random_state=0), hgb()),
    "oversampling": make_pipeline(RandomOverSampler(random_state=0), hgb()),
    "SMOTE": make_pipeline(SimpleImputer(strategy="median", add_indicator=True), SMOTE(random_state=0), hgb()),
    "class weights": make_pipeline(hgb("balanced")),
}
res_boost = evaluate(boosting)
res_boost

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (title, res) in zip(axes, [("logistic regression", res_linear), ("gradient boosting", res_boost)]):
    res[["neu_precision", "neu_recall", "macro_f1"]].plot.bar(ax=ax, rot=20)
    ax.set_title(title)
    ax.set_ylabel("score (time-based CV)")
plt.tight_layout()

## 3. Tasks

**Task 1.** Which strategy would you choose for the leaderboard, and why? Consider macro-F1, the neutral precision and recall, and the simplicity of the method.

**Task 2.** Show the wrong way: oversample `X, y` *before* the cross-validation and evaluate the boosting model. How much higher is the (false) score?

**Task 3.** Tune `sampling_strategy` of the undersampler, for example `{"pos": n}` for several `n`, with `GridSearchCV(..., cv=TimeSeriesSplit(5), scoring="f1_macro")`.

**Task 4 (optional).** Instead of resampling, fit the uncorrected boosting model and move the decision rule: predict "neu" whenever its predicted probability exceeds a threshold *t*. Choose *t* on the validation folds and compare with class weights (Session 8, decision thresholds).

In [ ]:
# TODO (Tasks 2-4)
